# The image manifold

Figure 20.29 of the book illustrates the manifold hypothesis: (a) an image of a handwritten digit, (b) a random
image of the same size, and (c) the digit rotated through 360 degrees, projected onto the first two principal
components. The book has the figure as images only; this notebook recreates it.

A $64 \times 64$ binary image is a point in a 4096-dimensional space with $2^{4096}$ possible points, but almost none
of them look like a digit: a random image (b) is noise. Images of digits occupy a tiny, structured subset. Rotating
one digit changes a single degree of freedom, so the rotated images trace a closed curve, a one-dimensional manifold,
through the 4096-dimensional space, and even a linear projection onto two principal components shows it as a
circle (c).

In [ ]:
import numpy as np
from matplotlib import pyplot as plt
from matplotlib.offsetbox import AnnotationBbox, OffsetImage
from scipy import ndimage
from tensorflow.keras.datasets import mnist

try:
    import probml_utils as pml
except ModuleNotFoundError:
    %pip install -qq git+https://github.com/probml/probml-utils.git
    import probml_utils as pml

## A digit and a random image

The book's digit is a binary 6 of about $64 \times 57$ pixels. We take a 6 from MNIST (training image 90, chosen
below), scale it from $28 \times 28$ to $64 \times 64$ and threshold it; the random image has the same fraction of
black pixels.

In [ ]:
(x_train, y_train), _ = mnist.load_data()
assert y_train[90] == 6
six = x_train[90] / 255.0
size = 64
gray = np.clip(ndimage.zoom(six, size / six.shape[0], order=1), 0, 1)
digit = (gray > 0.5).astype(float)

rng = np.random.default_rng(0)
noise = (rng.random(digit.shape) < digit.mean()).astype(float)
print(f"{size} x {size} = {size * size} pixels, {digit.mean():.0%} black")

In [ ]:
def show_binary(ax, img, title):
    ax.imshow(img, cmap="gray_r", vmin=0, vmax=1, interpolation="nearest")
    ax.set_xticks(np.arange(-0.5, img.shape[1], 2), minor=True)  # a dotted pixel grid, as in the book
    ax.set_yticks(np.arange(-0.5, img.shape[0], 2), minor=True)
    ax.grid(which="minor", color="0.6", ls=":", lw=0.5)
    ax.tick_params(which="both", length=0, labelbottom=False, labelleft=False)
    ax.set_title(title, loc="left", fontweight="bold")


for img, name in [(digit, "a"), (noise, "b")]:
    fig, ax = plt.subplots(figsize=(4.5, 4.5))
    show_binary(ax, img, f"({name})")
    fig.tight_layout()
    pml.savefig(f"image_manifold_{'digit' if name == 'a' else 'random'}.pdf")
    plt.show()

## Rotating the digit

360 rotations in steps of one degree. The grey-level digit is padded to $96 \times 96$ first so its corners are not
cut off, then each rotated image is flattened to a vector of $96^2 = 9216$ pixels, and PCA projects the 360 vectors
onto their first two principal components.

**Which circle the first two components show depends on the digit.** For the rotations of one image, the principal
components come in pairs of equal variance, and each pair traces a circle that winds round $k$ times per full
rotation, for one of the image's angular frequencies $k$; the first pair belongs to the strongest one. For the
first fifteen 6s of the MNIST training set it winds round 1, 2, 3 or 4 times; training image 90 winds round once,
as in the book's figure, so each point of the circle is one orientation of the digit. The cell checks this.

In [ ]:
pad = 16
padded = np.pad(gray, pad)
angles = np.arange(360)
rotated = np.stack([np.clip(ndimage.rotate(padded, a, reshape=False, order=1), 0, 1) for a in angles])
V = rotated.reshape(len(angles), -1)  # [360, D]
D = V.shape[1]

Vc = V - V.mean(axis=0)
U, S, Wt = np.linalg.svd(Vc, full_matrices=False)
Z = Vc @ Wt[:2].T  # coordinates on the first two principal components
explained = S**2 / np.sum(S**2)
print(f"D = {D} dimensions; the first two principal components explain {explained[:2].sum():.0%} of the variance")
print(f"variance of PC1, PC2, PC3: {explained[0]:.1%}, {explained[1]:.1%}, {explained[2]:.1%} (PCs come in equal pairs)")
r = np.linalg.norm(Z, axis=1)
print(f"distance from the centre: mean {r.mean():.2f}, relative spread {r.std() / r.mean():.1%}")
phi = np.unwrap(np.arctan2(Z[:, 1], Z[:, 0]))
winding = round((phi[-1] - phi[0] + (phi[1] - phi[0])) / (2 * np.pi))
print(f"the circle winds round {abs(winding)} time(s) per full rotation of the digit")
assert abs(winding) == 1

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6.5))
ax.plot(Z[:, 0], Z[:, 1], "x", color="#e34948", ms=6, mew=1.5)
for a in angles[::20]:  # a thumbnail every 20 degrees, just outside the circle
    pos = 1.28 * Z[a]
    thumb = OffsetImage(1 - rotated[a][pad // 2 : -pad // 2, pad // 2 : -pad // 2], cmap="gray", zoom=0.45)
    ax.add_artist(AnnotationBbox(thumb, pos, frameon=False))
lim = 1.5 * np.abs(Z).max()
ax.set_xlim(-lim, lim)
ax.set_ylim(-lim, lim)
ax.set_aspect("equal")
ax.set_xlabel("PC no 1")
ax.set_ylabel("PC no 2")
ax.set_title("(c)", loc="left", fontweight="bold")
fig.tight_layout()
pml.savefig("image_manifold_rotations.pdf")
plt.show()

## How flat is the manifold?

The rotated images lie on a one-dimensional curve, but the curve bends through many dimensions: the variance is
spread over many principal components, so a linear method needs far more than one component to reconstruct the
images, while one number, the angle, describes them all. This is the motivation for the nonlinear methods of
section 20.4 ([manifold_swiss_sklearn.ipynb](manifold_swiss_sklearn.ipynb)).

In [ ]:
cum = np.cumsum(explained)
for q in [0.5, 0.9, 0.99]:
    print(f"{np.searchsorted(cum, q) + 1:3d} principal components for {q:.0%} of the variance")

fig, ax = plt.subplots(figsize=(6, 3.5))
k = np.arange(1, 61)
ax.plot(k, cum[:60], "o-", color="#2a78d6", ms=3, lw=1.5)
ax.set_xlabel("number of principal components")
ax.set_ylabel("fraction of variance")
ax.set_ylim(0, 1.02)
fig.tight_layout()
pml.savefig("image_manifold_variance.pdf")
plt.show()